# G04 — Thesis-derived ensemble comparison (seed 42)

**Primary four families:** optimized weighted ensemble, fitted logistic stacking, SimCLR-anchored stacking (no bias), and CalStack-SSL with a frozen class-bias vector. The same family types are replayed in both image-aware and lesion-aware tracks. Two weighted-anchor variants and the original R07 class-bias selection are reported **only as ablations**, not included in primary family ranking.

**Required Kaggle inputs:** completed `g03a-image-level-all-ensembles-s42` and `g03b-lesion-aware-all-ensembles-s42`, both containing historical fitted predictions. **Optional inputs:** original `confidence-gated-ensemble` output for all 2,534 J1 test images; original `lesion-r07-n01-all-ensembles-s42` output for all 2,186 N01 validation images. CPU / Internet OFF.

**Limitations:** evaluation is source-stratified *within ISIC 2019*, not genuinely external HAM/BCN. Original J1 test and N01 validation scores have been examined; selecting a family here is exploratory and must be confirmed on independent data. No N01 locked test is touched.

## Configuration, numeric and provenance guards

In [1]:
import os,json,hashlib,time
from pathlib import Path
from collections import OrderedDict
import numpy as np,pandas as pd
from scipy.special import softmax
from sklearn.metrics import f1_score,log_loss

SEED=42
CLASSES=['MEL','NV','BCC','AK','BKL','DF','VASC','SCC']
INPUT=Path(os.getenv('G04_INPUT_DIR','/kaggle/input' if Path('/kaggle/input').exists() else '/mnt/data'))
WORK=Path(os.getenv('G04_WORK_DIR','/kaggle/working' if Path('/kaggle/working').exists() else '/mnt/data'))
OUT=WORK/'g04-thesis-family-comparison-s42'
for folder in ['tables','reports','predictions']:(OUT/folder).mkdir(parents=True,exist_ok=True)
START=time.monotonic();MAX_HOURS=2

def req(ok,msg):
    if not bool(ok):raise RuntimeError(msg)
def digest(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1<<20),b''):h.update(b)
    return h.hexdigest()
def jwrite(p,x):
    p=Path(p);p.parent.mkdir(parents=True,exist_ok=True)
    t=p.with_suffix(p.suffix+'.tmp');t.write_text(json.dumps(x,indent=2,allow_nan=False,default=str));t.replace(p)
def textarr(z,key):
    a=z[key];req(a.ndim==1 and a.dtype.kind in 'USO',f'{key}: invalid text-array layout')
    if a.dtype.kind=='O':req(all(isinstance(v,(str,np.str_)) for v in a),f'{key}: unexpected pickle objects')
    return a.astype(str)
def prob(p):
    a=np.asarray(p,dtype=np.float64)
    req(a.ndim==2 and a.shape[1]==8 and np.isfinite(a).all() and (a>=0).all(),f'Invalid probability array: {a.shape}')
    req(np.max(abs(a.sum(1)-1))<2e-3,'Probability rows do not sum to 1')
    a=np.maximum(a,1e-12);return a/a.sum(1,keepdims=True)
def blend(a,b,alpha):return prob((1-alpha)*a+alpha*b)
def classbias(p,b):
    b=np.asarray(b,dtype=np.float64)
    req(b.shape==(8,) and np.isfinite(b).all(),'Invalid 8-class bias vector')
    return prob(softmax(np.log(np.clip(p,1e-12,1))+b[None,:],axis=1))

def load(stage):
    hits=[]
    for f in INPUT.rglob('final_status.json'):
        try:
            status=json.loads(f.read_text())
            if status.get('stage')!=stage or status.get('status')!='COMPLETED':continue
            root=f.parent.parent;arc=root/'predictions/ensemble_probabilities.npz';spec=root/'reports/all_candidate_specs.json'
            if arc.is_file() and spec.is_file():hits.append((f,arc,spec,status))
        except (OSError,ValueError):pass
    req(len(hits)==1,f'{stage}: attach exactly one completed G03 output with prediction archive and candidate specs; found {len(hits)}')
    fp,ap,sp,status=hits[0]
    with np.load(ap,allow_pickle=True) as z:
        req(set(['image_id','y','source','group_id','classes','candidate_names','proba']).issubset(z.files),f'{stage} missing NPZ fields')
        ids=textarr(z,'image_id');classes=textarr(z,'classes');names=textarr(z,'candidate_names');source=textarr(z,'source');group=textarr(z,'group_id')
        req(classes.tolist()==CLASSES,f'{stage}: class order mismatch')
        req('y' in z and z['y'].dtype.kind in 'iu' and z['proba'].dtype.kind=='f',f'{stage}: predictions not numeric')
        y=z['y'].astype('int64');a=z['proba']
        req(a.shape==(len(names),len(ids),8),f'{stage}: wrong prediction shape')
        req(len(set(ids))==len(ids) and len(set(names))==len(names),'Nonunique image/method names')
        req(len(ids)==status['n_images'] and len(names)==status['candidate_count'],f'{stage}: output-status disagreement')
        req(np.isin(y,np.arange(8)).all() and set(source)=={'HAM10000','BCN20000'},f'{stage}: invalid labels or sources')
        ix={name:i for i,name in enumerate(names)}
        req(np.isfinite(a).all() and (a>=0).all() and np.max(abs(a.sum(2)-1))<.002,'Invalid predicted probabilities')
        arrays={name:a[i].astype(np.float64) for name,i in ix.items()}
    specs=json.loads(sp.read_text());req(set(specs)==set(names),f'{stage}: frozen recipe specs mismatch')
    meta=pd.DataFrame({'image_id':ids,'y':y,'source':source,'group_id':group})
    return {'meta':meta,'p':arrays,'specs':specs,'status':status,'root':ap.parent.parent,
            'provenance':{'stage':stage,'status_sha256':digest(fp),'archive_sha256':digest(ap),
                          'specs_sha256':digest(sp),'n_images':len(ids),'n_candidates':len(names)}}



## Load frozen G03 predictions

In [2]:
A=load('G03-A');B=load('G03-B')
req(A['status'].get('historical_replay',{}).get('available',False),'G03-A missing original fitted thesis stacker. Attach G03-A executed with original historical configuration/joblib.')
req(B['status'].get('source_reproduction_quality')=='CLOSE','G03-B source reproduction not CLOSE; investigate before scoring')
req(B['status'].get('test_touched') is False,'G03-B was not an untouched-validation replay')
print('Loaded G03 archives:',{k:x['provenance'] for k,x in [('image-aware',A),('lesion-aware',B)]})

def take(data,name):
    req(name in data['p'],f'Missing frozen candidate: {name}')
    return prob(data['p'][name])



Loaded G03 archives: {'image-aware': {'stage': 'G03-A', 'status_sha256': 'f499c21450b8105661f7bade7f5d81d8e10e8142c4f5ccda78adcee69eb79210', 'archive_sha256': '07482f53bdd1f999012244cb6c1121bfa1a838feb756dd24c80d717893e554a3', 'specs_sha256': '06e90f197be35fc4812852f73e1c64f8f22434d7e98967cf9146202e82037a91', 'n_images': 2239, 'n_candidates': 121}, 'lesion-aware': {'stage': 'G03-B', 'status_sha256': '5ced9dc51c72d4b752227a6dc67955fd47a44989a7785ca391a50dfd79e36e68', 'archive_sha256': 'daeac2a3793cb2fde8fbb436fb5d47b180150fcf0a76fa86fad188f89f93b980', 'specs_sha256': 'a65f3e01958d5225f5f2533f3a96b7770ed87469730bbf6703ced3578d11b794', 'n_images': 2109, 'n_candidates': 205}}


## Build thesis ensemble families and verify class bias

In [3]:
# Original thesis validation-fitted class bias. No reoptimisation on G03 scores.
ORIG_WEIGHT='OriginalThesis_ConstrainedWeighted'
ORIG_STACK='OriginalThesis_StackOnly'
ORIG_BLEND='OriginalThesis_StackBlend_NoBias'
ORIG_BIAS='OriginalThesis_CalStack_ClassBias'
for k in [ORIG_WEIGHT,ORIG_STACK,ORIG_BLEND,ORIG_BIAS]:req(k in A['p'],f'Missing exact original thesis candidate: {k}')
original_spec=A['specs'][ORIG_BIAS]
req(original_spec.get('kind')=='historical_class_bias','Historical thesis bias candidate specification is unexpected')
orig_bias=np.asarray(original_spec['bias'],dtype=float)
req(np.max(abs(classbias(take(A,ORIG_BLEND),orig_bias)-take(A,ORIG_BIAS)))<.002,'G03-A thesis CalStack vs bias-parent replay mismatch')
alpha_original=float(A['specs'][ORIG_BLEND]['alpha'])
req(np.isclose(alpha_original,.6,atol=1e-6),f'Unexpected thesis anchor alpha: {alpha_original}')

B_WEIGHT='Weighted_AllConstrained_1'
B_STACK='Stack_extended_balanced'
B_SIMCLR_BLEND=f'Blend_{B_STACK}__Cal_simclr__a{alpha_original:g}'
B_WEIGHT_BLEND=f'Blend_{B_STACK}__{B_WEIGHT}__a0.4'
for k in [B_WEIGHT,B_STACK,B_SIMCLR_BLEND,B_WEIGHT_BLEND]:req(k in B['p'],f'Missing R07 frozen counterpart: {k}')
req(B['specs'][B_WEIGHT]['spec']['kind']=='weighted','R07 constrained-weight candidate not frozen weighted')
req(B['specs'][B_STACK]['spec']['kind']=='stack','R07 logistic-stack candidate not a fitted stack')
req(B['specs'][B_SIMCLR_BLEND]['spec']['kind']=='blend','R07 SimCLR anchor counterpart is not a blend')
req(np.max(abs(blend(take(B,'Cal_simclr'),take(B,B_STACK),alpha_original)-take(B,B_SIMCLR_BLEND)))<.002,
    'R07 SimCLR anchor blend is not reproduced from its frozen components')

# Two independent notions of class bias:
# transferred: exact historical thesis bias applied to the common SimCLR-anchored architecture;
# historical: each original study's selected, fitted bias (the R07 anchor may differ).
# The historical R07 bias was selected on the R07 validation cohort and is exploratory.
selected=B['status'].get('original_r07_selected_method','')
biased=[name for name,s in B['specs'].items() if s.get('kind')=='class_bias' and s.get('spec',{}).get('kind')=='class_bias']
anchored_biased=[name for name in biased if B['specs'].get(B['specs'][name]['spec'].get('parent',''),{}).get('spec',{}).get('kind')=='blend']
if selected in anchored_biased:B_BIAS=selected
elif len(anchored_biased)==1:B_BIAS=anchored_biased[0]
else:B_BIAS=None
if B_BIAS is None:print('R07 anchored class-bias candidate absent or nonunique; historical track-specific row omitted')
B_BIAS_PARENT=None
if B_BIAS:
    bias_spec=B['specs'][B_BIAS]['spec'];B_BIAS_PARENT=bias_spec['parent']
    req(B_BIAS_PARENT in B['p'],f'R07 class-bias parent missing: {B_BIAS_PARENT}')
    req(np.max(abs(classbias(take(B,B_BIAS_PARENT),bias_spec['offsets'])-take(B,B_BIAS)))<.002,
        'R07 original class bias vs parent replay mismatch')

COMMON=OrderedDict([
 ('OptimizedWeighted',(ORIG_WEIGHT,B_WEIGHT)),
 ('LogisticStacking',(ORIG_STACK,B_STACK)),
 ('SimCLRAnchor_NoBias',(ORIG_BLEND,B_SIMCLR_BLEND)),
 ('SimCLRAnchor_ClassBias_OriginalVector',(ORIG_BIAS,B_SIMCLR_BLEND)),
 ('WeightedAnchor_NoBias_alpha0.4',(None,B_WEIGHT_BLEND)),
 ('WeightedAnchor_ClassBias_OriginalVector',(None,B_WEIGHT_BLEND)),
])
P=OrderedDict([('image_aware',OrderedDict()),('lesion_aware',OrderedDict())]);METHODS={};EXTRA={}
for family,(origname,newname) in COMMON.items():
    if family=='SimCLRAnchor_ClassBias_OriginalVector':
        pa=take(A,ORIG_BIAS);pb=classbias(take(B,B_SIMCLR_BLEND),orig_bias)
    elif family=='WeightedAnchor_NoBias_alpha0.4':
        pa=blend(take(A,ORIG_WEIGHT),take(A,ORIG_STACK),.4);pb=take(B,B_WEIGHT_BLEND)
    elif family=='WeightedAnchor_ClassBias_OriginalVector':
        pa=classbias(blend(take(A,ORIG_WEIGHT),take(A,ORIG_STACK),.4),orig_bias)
        pb=classbias(take(B,B_WEIGHT_BLEND),orig_bias)
    else:pa=take(A,origname);pb=take(B,newname)
    P['image_aware'][family]=pa;P['lesion_aware'][family]=pb
    METHODS[family]={'image_aware':origname or 'Derived from frozen historical components',
                    'lesion_aware':newname,'bias_provenance':'ORIGINAL_THESIS_VALIDATION' if 'ClassBias' in family else 'NONE',
                    'same_inputs_across_protocols':False,'same_family':True}

if B_BIAS:
    P['image_aware']['HistoricalTrackSpecific_CalStackClassBias']=take(A,ORIG_BIAS)
    P['lesion_aware']['HistoricalTrackSpecific_CalStackClassBias']=take(B,B_BIAS)
    METHODS['HistoricalTrackSpecific_CalStackClassBias']={'image_aware':ORIG_BIAS,'lesion_aware':B_BIAS,
        'lesion_aware_parent':B_BIAS_PARENT,'bias_provenance':'THESIS_VALIDATION_AND_R07_VALIDATION',
        'same_inputs_across_protocols':False,'same_family':True,
        'comparability':'Historical methods use different anchors and R07 class bias was validation-optimised'}

req(len(P['image_aware'])==len(P['lesion_aware']),'Candidate catalogue differs by protocol')
req(all(P[t][name].shape==(len(D['meta']),8) for t,D in [('image_aware',A),('lesion_aware',B)] for name in P[t]),
    'Ensemble outputs have invalid row counts')




## Score pooled ISIC source subsets, HAM, BCN

In [4]:
def ece(y,p,bins=15):
    pred=p.argmax(1);conf=p.max(1);v=0.
    for i in range(bins):
        m=(conf>=i/bins)&(conf<((i+1)/bins if i<bins-1 else 1+1e-8))
        if m.any():v+=m.mean()*abs((pred[m]==y[m]).mean()-conf[m].mean())
    return float(v)
def metrics(y,p):
    y=np.asarray(y,dtype=int);p=prob(p);pred=p.argmax(1);classes=np.unique(y)
    req(len(y)>0 and len(classes)>=2,'Need at least two observed classes')
    return {'n':len(y),'n_classes':len(classes),'balanced_accuracy':float(np.mean([(pred[y==c]==c).mean() for c in classes])),
        'accuracy':float(np.mean(pred==y)),
        'macro_f1':float(f1_score(y,pred,labels=classes,average='macro',zero_division=0)),
        'nll':float(log_loss(y,p,labels=np.arange(8))), 'ece15':ece(y,p),
        'unsupported_rate':float(np.mean(~np.isin(pred,classes)))}

ROWS=[];RECALL=[]
for track,data in [('image_aware',A),('lesion_aware',B)]:
    meta=data['meta'];y=meta.y.to_numpy(dtype=int);source=meta.source.to_numpy(dtype=str)
    for family,p in P[track].items():
        for cohort,ix in [('ISIC2019_source_pooled',np.arange(len(meta))),
                          ('HAM10000',np.flatnonzero(source=='HAM10000')),
                          ('BCN20000',np.flatnonzero(source=='BCN20000'))]:
            yy=y[ix];pp=p[ix];m=metrics(yy,pp)
            ROWS.append({'protocol':track,'cohort':cohort,'family':family,**m})
            pred=pp.argmax(1)
            for c in np.unique(yy):RECALL.append({'protocol':track,'cohort':cohort,'family':family,
                'class':CLASSES[c],'count':int(np.sum(yy==c)),'recall':float(np.mean(pred[yy==c]==c))})
metric_table=pd.DataFrame(ROWS)
metric_table.to_csv(OUT/'tables/all_thesis_family_metrics.csv',index=False)
pd.DataFrame(RECALL).to_csv(OUT/'tables/per_class_recall.csv',index=False)

rank=metric_table.loc[metric_table.cohort.ne('ISIC2019_source_pooled'),['protocol','cohort','family','balanced_accuracy']].copy()
rank['column']=rank.protocol+'_'+rank.cohort
rank=rank.pivot(index='family',columns='column',values='balanced_accuracy')
expect=['image_aware_HAM10000','image_aware_BCN20000','lesion_aware_HAM10000','lesion_aware_BCN20000']
req(set(rank.columns)==set(expect) and rank[expect].notna().all().all(),'Missing any of the four source evaluations')
rank=rank[expect];rank['mean_bacc']=rank[expect].mean(1);rank['worst_bacc']=rank[expect].min(1)
rank['range_bacc']=rank[expect].max(1)-rank[expect].min(1)
rank['harmonic_bacc']=len(expect)/(1/np.clip(rank[expect],1e-9,None)).sum(1)
PRIMARY={'OptimizedWeighted','LogisticStacking','SimCLRAnchor_NoBias','SimCLRAnchor_ClassBias_OriginalVector'}
rank['eligible_for_preliminary_family_ranking']=[k in PRIMARY for k in rank.index]
rank=rank.sort_values(['eligible_for_preliminary_family_ranking','mean_bacc','worst_bacc'],ascending=[False,False,False])
rank.to_csv(OUT/'tables/family_rankings_four_source_scores.csv')
eligible=rank.loc[rank.eligible_for_preliminary_family_ranking]
provisional=str(eligible.index[0]);delta=float(eligible.iloc[0].mean_bacc-eligible.iloc[1].mean_bacc)

# Supplementary exact-image overlap report; no invented pairing between distinct J1/N01 samples.
m=A['meta'].merge(B['meta'],on='image_id',suffixes=('_original','_n01'),how='inner',validate='one_to_one')
req((m.y_original==m.y_n01).all() and (m.source_original==m.source_n01).all(),
    'Two protocols disagree on diagnosis/source for a shared image ID')
m[['image_id','y_original','source_original']].to_csv(OUT/'reports/identical_image_overlap.csv',index=False)



## Optional full ISIC 2019 historical replays

In [5]:
# Optional original full ISIC test, when exact original thesis output is provided.
FULL=[];REPLAY_AUDIT=[]
for p in INPUT.rglob('final_internal_probabilities.npz'):
    try:
        with np.load(p,allow_pickle=True) as z:
            needed=['y_test','weighted_test_prob','stacking_test_prob','stacking_bias_test_prob']
            if not set(needed).issubset(z.files):continue
            y=np.asarray(z['y_test'],dtype=int)
            req(len(y)==2534,'Original thesis full test must have 2,534 images')
            for family,key in [('OptimizedWeighted','weighted_test_prob'),
                               ('SimCLRAnchor_NoBias','stacking_test_prob'),
                               ('SimCLRAnchor_ClassBias_OriginalVector','stacking_bias_test_prob')]:
                FULL.append({'protocol':'image_aware','cohort':'ISIC2019_J1_full_test','family':family,**metrics(y,z[key])})
    except Exception as err:raise RuntimeError(f'Invalid original thesis full-ISIC artifact {p}: {err}')
    break

# Optional complete N01 validation replay of the same frozen candidate family (not a test).
r07=[]
for p in INPUT.rglob('final_ensemble_config.json'):
    try:
        cfg=json.loads(p.read_text())
        if cfg.get('stage')=='R07' and cfg.get('status')=='COMPLETED':r07.append((p,cfg))
    except (OSError,ValueError):pass
req(len(r07)<=1,'Multiple R07 frozen configs attached; remove duplicate Kaggle inputs')
if r07:
    cp,cfg=r07[0];rroot=cp.parent.parent
    req(cfg.get('n01_manifest_sha256')==B['status'].get('n01_manifest_sha256'),'R07/N01 manifest does not match G03-B')
    val=rroot/'predictions/final_validation.npz'
    req(val.is_file(),f'Missing full R07 validation labels: {val}')
    with np.load(val,allow_pickle=True) as z:
        ids=textarr(z,'image_id');y=np.asarray(z['y'],dtype=int)
        req(len(ids)==len(set(ids))==2186,'Invalid full R07 validation identities')
        req(textarr(z,'classes').tolist()==CLASSES if 'classes' in z else True,'R07 full validation class mismatch')
        look=pd.Index(ids).get_indexer(B['meta'].image_id)
        req((look>=0).all() and np.array_equal(y[look],B['meta'].y.to_numpy()),'Full R07 validation identities/labels do not align with G03-B')
        def rprob(n):
            path=rroot/'predictions'/f'candidate_{n}.npy'
            req(path.is_file(),f'Missing R07 candidate prediction: {path.name}')
            a=np.load(path,allow_pickle=False);req(a.shape==(len(ids),8),'R07 full candidate shape mismatch')
            return prob(a)
        for family,(_,bname) in COMMON.items():
            if family=='SimCLRAnchor_ClassBias_OriginalVector':p=classbias(rprob(B_SIMCLR_BLEND),orig_bias)
            elif family=='WeightedAnchor_ClassBias_OriginalVector':p=classbias(rprob(B_WEIGHT_BLEND),orig_bias)
            else:p=rprob(bname)
            difference=np.abs(p[look]-P['lesion_aware'][family])
            quality='CLOSE' if float(difference.mean())<.004 and float(np.percentile(difference,99))<.035 else 'REVIEW'
            REPLAY_AUDIT.append({'family':family,'mean_abs_difference':float(difference.mean()),
                'p99_abs_difference':float(np.percentile(difference,99)),'max_abs_difference':float(difference.max()),'quality':quality})
            if quality!='CLOSE':print('REVIEW R07 full-validation replay:',family,'mean=',round(float(difference.mean()),5))
            FULL.append({'protocol':'lesion_aware','cohort':'ISIC2019_N01_full_validation','family':family,
                         'replay_quality':quality,**metrics(y,p)})
        if B_BIAS:
            p=rprob(B_BIAS)
            FULL.append({'protocol':'lesion_aware','cohort':'ISIC2019_N01_full_validation',
                        'family':'HistoricalTrackSpecific_CalStackClassBias',**metrics(y,p)})
if FULL:pd.DataFrame(FULL).to_csv(OUT/'tables/optional_full_isic2019_metrics.csv',index=False)
else:pd.DataFrame(columns=['protocol','cohort','family','balanced_accuracy']).to_csv(OUT/'tables/optional_full_isic2019_metrics.csv',index=False)



## Export four-family ranking and audit records

In [6]:
pd.DataFrame(REPLAY_AUDIT).to_csv(OUT/'reports/full_r07_replay_audit.csv',index=False)
jwrite(OUT/'reports/frozen_family_recipes.json',METHODS)
jwrite(OUT/'reports/final_status.json',{'stage':'G04-THESIS-FAMILY','status':'COMPLETED','seed':SEED,
    'source_archives':[A['provenance'],B['provenance']], 'n_candidate_families':len(rank),
    'source_only_exploratory_provisional_family':provisional,
    'runner_up_mean_difference_pp':delta*100,
    'evaluation_was_used_for_family_ranking':True,'N01_locked_test_touched':False,
    'orig_thesis_bias':orig_bias.tolist(),'historical_r07_class_bias_candidate':B_BIAS,
    'historical_r07_class_bias_parent':B_BIAS_PARENT,
    'n_identical_images_in_two_evaluation_subsets':len(m),
    'source_interpretation':'HAM/BCN source-stratified ISIC2019 subsets, not external source-excluded tests',
    'selection':'EXPLORATORY ONLY; independent locked test and genuinely source-excluded transfer needed'})
jwrite(OUT/'reports/EXPLORATORY_PROVISIONAL_NOT_FINAL.json',{'family':provisional,
    'mean_source_bacc':float(eligible.iloc[0].mean_bacc),'worst_source_bacc':float(eligible.iloc[0].worst_bacc),
    'same_formula_across_track':'Same named family; calibrated model sources/stackers and fitted coefficients differ',
    'not_for_final_paper_claim':True})
print('\nTHESIS-FAMILY CROSS-DATASET BALANCED ACCURACY (%)')
print((100*rank[expect+['mean_bacc','worst_bacc']]).round(2).to_string())
print('\nExploratory best family:',provisional,'| margin over runner-up:',round(100*delta,3),'pp')
print('Optional full ISIC rows',len(FULL),'| exact overlapping G03 image IDs',len(m))
print('Files saved:',OUT)



THESIS-FAMILY CROSS-DATASET BALANCED ACCURACY (%)
column                                     image_aware_HAM10000  image_aware_BCN20000  lesion_aware_HAM10000  lesion_aware_BCN20000  mean_bacc  worst_bacc
family                                                                                                                                                    
SimCLRAnchor_NoBias                                       81.74                 82.68                  65.86                  45.71      69.00       45.71
OptimizedWeighted                                         77.54                 81.38                  65.69                  48.24      68.21       48.24
SimCLRAnchor_ClassBias_OriginalVector                     81.44                 82.50                  64.82                  44.02      68.20       44.02
LogisticStacking                                          82.89                 81.77                  61.72                  45.51      67.97       45.51
WeightedAnchor_NoBi

## Interpretation notes

The original thesis used **seven** source predictions (including SimCLR TTA), whereas N01 uses **six**; meta-stacker inputs and fit data differ. The comparison therefore matches the **ensemble family**, not every model input. Original thesis CalStack used anchor `SimCLR`, stacking fraction `0.6`, and an eight-dimensional validation-optimized class-bias vector. For the aligned class-bias ablation, those same frozen offsets are applied to the R07 SimCLR-anchored no-bias output; no offsets are refit on HAM/BCN. R07's own selected class-bias is also replayed in a *separate* historical row only if its parent is a fitted stack blend. This was chosen on R07 validation and must not be treated as independent validation. Full ISIC 2019 and HAM/BCN source subsets overlap, so their scores are not added together when ranking.

Output `reports/EXPLORATORY_PROVISIONAL_NOT_FINAL.json` is **not** a proposed journal model until confirmed on held-out data. The 2,244-image N01 locked test must remain untouched during selection.